# Intelligent Agents: Reflex-Based Agents for GridHunt (Multi-agent Environment)

Student Name: [Add your name]

I have used the following AI tools: [list tools]

I understand that my submission needs to be my own work: [your initials]

## Learning Outcomes

* Apply core AI concepts by implementing the agent function for a simple and model-based reflex agents that respond to environmental percepts.
* Practice how the environment and the agent function interact.
* Analyze agent performance through controlled experiments across different environment configurations.

## Instructions

Total Points: 10

Complete this notebook. Use the provided notebook cells and insert additional code and markdown cells as needed. Submit the completely rendered notebook.
### AI Use

Here are some guidelines that will make it easier for you:

* __Don't:__ Rely on AI auto completion. You will waste a lot of time trying to figure out how the suggested code relates to what we do in class. Turn off AI code completion (e.g., Copilot) in your IDE.
* __Don't:__ Do not submit code/text that you do not understand or have not checked to make sure that it is complete and correct.
* __Do:__ Use AI for debugging and letting it explain code and concepts from class.

## Introduction

![GridHunt Image](gridhunt.png)

Gridhunt is a small educational game that helps you practice implementing simple agent functions. It is a modified Python reimplementation of Gridhunt2. 

The objective of gridhunt is to implement an intelligent agent, the hunter, who can catch a monster faster than all other hunters. Gridhunt uses a $n \times n$ arena consisting of a grid of tiles. Gridhunt is a turn-based game, and several hunter agents and the monster agent move around in the arena. A hunter catches the monster if she/he moves on the same square the monster currently occupies. The hunters compete fow who catches the monster first. If the monster survives the maximum number of steps for the game, then the monster wins.

## PEAS Description of Gridhunt

__Performance Measure:__ The performance is measured as the number of steps the hunter uses to catch the monster. Catching the monster means to be on the same square.

__Environment:__ An arena with $n \times n$ squares. At the beginning the monster and the hunters are randomly placed in the arena environment. The hunters and the monster
    can move around, but cannot leave the arena. Since there are three agents involved, this is a competitive multi-agent environment.

__Actuators:__ The agent can move to an adjacent square using actions "north", "east", "west", and "south", or teleport which will move the agent 
    to a random square in the arena.

__Sensors:__ The agent can always see its location in the arena and the location of the monster.  

## The Arena Environment

The environment manages the location of the agents (hunter and monster). It initially places them in a random location and then provides them in every step with percepts (the location of the hunter and the monster) and asks them for their action. 

__A Note on positions:__
The arena is implemented as an array with row and column indices representing the position.
Positions are stored in a numpy array where `pos[0]` is the row index and `pos[1]` is the column index in the arena. North means up in this array, that is the row index gets smaller when going north. Remember indices in Python start with 0 and go to `n-1`.

In [31]:
import numpy as np
from IPython.display import clear_output
from time import sleep

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def arena_environment(hunter_1_agent_function, hunter_2_agent_function, monster_agent_function, n = 30, max_steps = 100, visualize = False, animation = False):
    """
    Simulate an arena where two hunter agent tries to catch a monster agent.

    Parameters:
    hunter_1_agent_function and hunter_2_agent_function (function): A function that takes the current positions of the hunter and monster
                                      and returns the next move for the hunter ('north', 'east', 'west', 'south', 'teleport').
    monster_agent_function (function): A function that takes the current positions of the hunter and monster
                                       and returns the next move for the monster (or 'stay' to remain in place).
    n (int): The size of the arena (n x n grid).
    max_steps (int): The maximum number of steps to simulate.
    visualize (bool): Whether to visualize the arena.
    animation (bool): Whether to animate the visualization with a delay.

    Returns:
    int: The number of steps taken for the hunter to catch the monster or np.nan (not a number) if not caught.
    """
    
    # Initialize positions
    monster_pos = np.random.randint(0, n-1, size=2)
    hunter_1_pos = np.random.randint(0, n-1, size=2)
    while np.array_equal(hunter_1_pos, monster_pos):
        hunter_1_pos = np.random.randint(0, n-1, size=2)
    hunter_2_pos = np.random.randint(0, n-1, size=2)
    while np.array_equal(hunter_2_pos, monster_pos):
        hunter_2_pos = np.random.randint(0, n-1, size=2)    
    
    def move(action, position):
        """calculate new position for the agent."""
        if action == 'north':
            position[0] -= 1
        elif action == 'south':
            position[0] += 1
        elif action == 'west':
            position[1] -= 1
        elif action == 'east':
            position[1] += 1
        else:
            raise ValueError("Invalid action.")
        
        # Ensure position stays within bounds
        position = np.clip(position, 0, n-1)
        return position

    def print_arena():

        if animation:
            sleep(1)
            clear_output(wait=True)
        print(f"Step {step}:")
        print(f"Hunter 1 is at '{hunter_1_pos}'; Hunter 2 is at '{hunter_2_pos}'; Monster is at '{monster_pos}'\n")
        arena = np.full((n, n), '.', dtype=str)
        arena[monster_pos[0], monster_pos[1]] = 'M'
        arena[hunter_1_pos[0], hunter_1_pos[1]] = '1'
        arena[hunter_2_pos[0], hunter_2_pos[1]] = '2'
        print("\n".join("".join(row) for row in arena))

    for step in range(max_steps):
        if visualize:
            print_arena()
        
        # Check if hunter has caught the monster
        if np.array_equal(hunter_1_pos, monster_pos):
            if visualize:
                print(f"Hunter 1 caught the monster in {step} steps!")
            return ("1", step)
        
        if np.array_equal(hunter_2_pos, monster_pos):
            if visualize:
                print(f"Hunter 2 caught the monster in {step} steps!")
            return ("2", step)

        # Get next move from monster agent function
        monster_action = monster_agent_function(monster_pos, hunter_1_pos, hunter_2_pos)
        if monster_action != 'stay':
            monster_pos = move(monster_action, monster_pos)

        # Get next move from hunter agent function
        hunter_1_action = hunter_1_agent_function(hunter_1_pos, monster_pos)
        if hunter_1_action == 'teleport':
            hunter_1_pos = np.random.randint(0, n-1, size=2)
        else:
            hunter_1_pos = move(hunter_1_action, hunter_1_pos)

        hunter_2_action = hunter_2_agent_function(hunter_2_pos, monster_pos)
        if hunter_2_action == 'teleport':
            hunter_2_pos = np.random.randint(0, n-1, size=2)
        else:
            hunter_2_pos = move(hunter_2_action, hunter_2_pos)
    
        # print the agents' actions
        if visualize:
            print(f"Hunter 1 chose action '{hunter_1_action}'")
            print(f"Hunter 2 chose action '{hunter_2_action}'")
            print(f"Monster chose action '{monster_action}'")
            print("\n" + "="*n + "\n")

    # the hunter failed to catch the monster within max_steps
    if visualize:
        print(f"Monster won by surviving {max_steps} steps.")
    
    return ("M", np.nan)

I implement the monster here as a simple agent that moves around randomly, but mostly stays in its place. You can use AI to get a detailed explanation of how the following code works.

In [32]:
monster_actions = ["north", "east", "west", "south", "stay"]

def monster_agent_function_simple(monster_pos, hunter_1_pos, hunter_2_pos):
    return np.random.choice(monster_actions, p=[0.125, 0.125, 0.125, 0.125, 0.5])

# The Hunter Agent

Your job is to implement a simple-reflex hunter agent that can catch the monster. Remember, your agent is implemented as an agent function that get percepts and needs to return a valid action. The actions are: 

In [33]:
actions = ["north", "east", "west", "south", "teleport"]

## A Simple Example Implementation

Here is a very simple hunter that just runs around randomly and hopes that it bumps into the monster. 

In [34]:
def simple_randomized_hunter_agent_function(hunter_location, monster_location):
    return np.random.choice(actions)

Ask the agent for an action.

In [35]:
simple_randomized_hunter_agent_function([0,0], [5,5])

np.str_('teleport')

## Experimenting With the Agent

We can place the monster and the hunter into the environment and run a simulation by calling the environment function.

In [36]:
hunter_1 = simple_randomized_hunter_agent_function
hunter_2 = simple_randomized_hunter_agent_function

arena_environment(hunter_1, 
                  hunter_2, 
                  monster_agent_function_simple, 
                  n=5, max_steps=5, visualize=True, animation=False)

Step 0:
Hunter 1 is at '[1 0]'; Hunter 2 is at '[3 2]'; Monster is at '[0 2]'

..M..
1....
.....
..2..
.....
Hunter 1 chose action 'east'
Hunter 2 chose action 'east'
Monster chose action 'stay'

=====

Step 1:
Hunter 1 is at '[1 1]'; Hunter 2 is at '[3 3]'; Monster is at '[0 2]'

..M..
.1...
.....
...2.
.....
Hunter 1 chose action 'north'
Hunter 2 chose action 'teleport'
Monster chose action 'north'

=====

Step 2:
Hunter 1 is at '[0 1]'; Hunter 2 is at '[2 0]'; Monster is at '[0 2]'

.1M..
.....
2....
.....
.....
Hunter 1 chose action 'teleport'
Hunter 2 chose action 'east'
Monster chose action 'stay'

=====

Step 3:
Hunter 1 is at '[2 1]'; Hunter 2 is at '[2 1]'; Monster is at '[0 2]'

..M..
.....
.2...
.....
.....
Hunter 1 chose action 'west'
Hunter 2 chose action 'north'
Monster chose action 'stay'

=====

Step 4:
Hunter 1 is at '[2 0]'; Hunter 2 is at '[1 1]'; Monster is at '[0 2]'

..M..
.2...
1....
.....
.....
Hunter 1 chose action 'north'
Hunter 2 chose action 'east'
Monster c

('M', nan)

Well, this was one run, maybe it was just good or bad luck this time!

Let's run an experiment with 100 runs in a $20 \times 20$ arena.  

In [37]:
hunter_1 = simple_randomized_hunter_agent_function
hunter_2 = simple_randomized_hunter_agent_function

steps = [arena_environment(hunter_1, 
                           hunter_2, 
                           monster_agent_function_simple, 
                           n=20, max_steps=100, visualize=False) for _ in range(100)] 
print(steps)

[('M', nan), ('M', nan), ('M', nan), ('1', 95), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('2', 35), ('M', nan), ('2', 40), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('1', 80), ('M', nan), ('M', nan), ('2', 3), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('1', 4), ('1', 79), ('M', nan), ('2', 47), ('M', nan), ('M', nan), ('2', 39), ('M', nan), ('2', 5), ('2', 28), ('1', 44), ('2', 68), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('1', 56), ('M', nan), ('M', nan), ('M', nan), ('1', 66), ('M', nan), ('2', 95), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('2', 22), ('M', nan), ('M', nan), ('M', nan), ('2', 86), ('M', nan), ('1', 80), ('1', 98), ('M', nan), ('M', nan), ('2', 18), ('M', nan), ('M', nan), ('1', 9), ('M', nan), ('2', 32), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('2', 40), ('M', nan), ('1', 29), ('M', nan), ('1', 64), ('1', 2), ('M', nan), ('M', nan), ('M', nan), ('M', nan),

We just got a list with who one and the number of steps to catch the monster for 100 simulation runs.
Let's analysis the results by counting who won how often.

In [38]:
steps = np.array(steps)

print (f"Hunter 1 won: {np.sum(steps[:,0] == "1")}")
print (f"Hunter 2 won: {np.sum(steps[:,0] == "2")}")
print (f"Monster 1 won: {np.sum(steps[:,0] == "M")}")       


Hunter 1 won: 15
Hunter 2 won: 14
Monster 1 won: 71


The random agent in not a very good hunter. You need to implement a better agent function.

# Your Agent Implementation [4 points]

Write a new hunter agent function that chases the monster. Copy the simple randomized hunter function from above and modify it using rules based on the monster's and the hunter's location. Let your hunter compete with the random agent. You can also come up with several different agents and let them compete against each other.

In [39]:
# Chiến lược 1: Greedy Reflex Hunter (dùng khoảng cách Manhattan, luôn di chuyển về phía quái vật)
def greedy_hunter_agent(hunter_position, monster_position, verbose=False):
    dr = monster_position[0] - hunter_position[0]
    dc = monster_position[1] - hunter_position[1]

    if abs(dr) >= abs(dc):
        action = "south" if dr > 0 else "north"
    else:
        action = "east" if dc > 0 else "west"

    if verbose:
        print(f"[greedy] hunter={tuple(hunter_position)} monster={tuple(monster_position)} "
              f"dr={dr} dc={dc} -> action='{action}'")
    return action


# Chiến lược 2: Model-Based / Predictive Hunter (dùng Teleport khi ở quá xa quái vật)
def smart_hunter_agent_function(hunter_position, monster_position, arena_size=30, verbose=False):
    manhattan_dist = abs(hunter_position[0] - monster_position[0]) + abs(hunter_position[1] - monster_position[1])

    if manhattan_dist > arena_size * 0.7:
        if np.random.rand() < 0.3:
            if verbose:
                print(f"[smart] dist={manhattan_dist} > 70% of {arena_size} -> teleport roll succeeded")
            return "teleport"
        elif verbose:
            print(f"[smart] dist={manhattan_dist} > 70% of {arena_size} but teleport roll failed, moving normally")

    dr = monster_position[0] - hunter_position[0]
    dc = monster_position[1] - hunter_position[1]

    if abs(dr) >= abs(dc):
        action = "south" if dr > 0 else "north"
    else:
        action = "east" if dc > 0 else "west"

    if verbose:
        print(f"[smart] hunter={tuple(hunter_position)} monster={tuple(monster_position)} "
              f"dist={manhattan_dist} dr={dr} dc={dc} -> action='{action}'")
    return action


# Monster luôn di chuyển ra xa hunter gần mình nhất (max-min distance)
def monster_agent(monster_position, hunter_position_1, hunter_position_2, verbose=False):
    best_action = 'stay'
    max_min_distance = -1

    candidates = {
        'stay': monster_position,
        'north': np.array([max(0, monster_position[0] - 1), monster_position[1]]),
        'south': np.array([monster_position[0] + 1, monster_position[1]]),
        'west': np.array([monster_position[0], max(0, monster_position[1] - 1)]),
        'east': np.array([monster_position[0], monster_position[1] + 1])
    }

    scores = {}
    for act, pos in candidates.items():
        d1 = np.linalg.norm(pos - hunter_position_1)
        d2 = np.linalg.norm(pos - hunter_position_2)
        min_distance = min(d1, d2)
        scores[act] = min_distance
        if min_distance > max_min_distance:
            max_min_distance = min_distance
            best_action = act

    if verbose:
        print(f"[monster] pos={tuple(monster_position)} scores={ {k: round(v,2) for k,v in scores.items()} } "
              f"-> action='{best_action}' (max-min dist={max_min_distance:.2f})")
    return best_action


# Your Experiments [2 points]

Copy the simulation code from above and run experiments with your agent. 
Experiment with larger arenas of at least size $30 \times 30$.

In [40]:
class CooperativeHunterTeam:
    def __init__(self, arena_size=30, verbose=False):
        self.arena_size = arena_size
        self.verbose = verbose
        self.pos = {1: None, 2: None}

    def _direction(self, dr, dc):
        if abs(dr) >= abs(dc):
            return "south" if dr > 0 else "north"
        else:
            return "east" if dc > 0 else "west"

    def _act(self, me, hunter_pos, monster_pos):
        self.pos[me] = np.array(hunter_pos)
        other = self.pos[3 - me]

        if other is None:
            dr = monster_pos[0] - hunter_pos[0]
            dc = monster_pos[1] - hunter_pos[1]
            action = self._direction(dr, dc)
            if self.verbose:
                print(f"[team H{me}] no info on partner yet -> chase directly, action='{action}'")
            return action

        my_dist = abs(hunter_pos[0]-monster_pos[0]) + abs(hunter_pos[1]-monster_pos[1])
        other_dist = abs(other[0]-monster_pos[0]) + abs(other[1]-monster_pos[1])

        if my_dist <= other_dist:
            role = "chaser"
            dr = monster_pos[0] - hunter_pos[0]
            dc = monster_pos[1] - hunter_pos[1]
        else:
            role = "interceptor"
            flee_r = np.sign(monster_pos[0] - other[0])
            flee_c = np.sign(monster_pos[1] - other[1])
            target = np.clip(np.array([monster_pos[0] + flee_r*2, monster_pos[1] + flee_c*2]),
                              0, self.arena_size - 1)
            dr, dc = target[0] - hunter_pos[0], target[1] - hunter_pos[1]
            if dr == 0 and dc == 0:
                dr = monster_pos[0] - hunter_pos[0]
                dc = monster_pos[1] - hunter_pos[1]

        action = self._direction(dr, dc)
        if self.verbose:
            print(f"[team H{me}] role={role} my_dist={my_dist} other_dist={other_dist} -> action='{action}'")
        return action

    def hunter_1_act(self, hunter_pos, monster_pos):
        return self._act(1, hunter_pos, monster_pos)

    def hunter_2_act(self, hunter_pos, monster_pos):
        return self._act(2, hunter_pos, monster_pos)

def run_exp(h1_fn, h2_fn, monster_fn, arena_size=30, num_runs=100, max_steps=100, verbose=False):
    results = {"1": 0, "2": 0, "M": 0}
    steps_list = []
    for i in range(num_runs):
        winner, steps = arena_environment(
            h1_fn, h2_fn, monster_fn,
            n=arena_size, max_steps=max_steps, visualize=False
        )
        results[winner] += 1
        if not np.isnan(steps):
            steps_list.append(steps)

        if verbose:
            print(f"run {i+1}/{num_runs}: winner='{winner}', steps={steps}")

    avg_steps = np.mean(steps_list) if steps_list else np.nan
    return results, avg_steps

# 1. Experiment 1: Greedy Hunter vs Random Hunter (Arena 20x20)
exp1_res, exp1_steps = run_exp(
    greedy_hunter_agent([2, 3], [5, 1], verbose=True), 
    simple_randomized_hunter_agent_function, 
    monster_agent_function_simple, 
    smart_hunter_agent_function([2, 3], [5, 1], arena_size=20, verbose=True),
    arena_size=20, num_runs=100
)


# 2. Experiment 2: Smart Hunter vs Greedy Hunter (Arena 30x30)
team = CooperativeHunterTeam(arena_size=10, verbose=True)
exp2_res, exp2_steps = run_exp(
    lambda h, m: smart_hunter_agent_function(h, m, 30), 
    greedy_hunter_agent, 
    monster_agent_function_simple, 
    arena_size=30, num_runs=100
)

# 3. Experiment 3: Smart Hunter vs Smart Monster (Arena 40x40)
exp3_res, exp3_steps = run_exp(
    lambda h, m: smart_hunter_agent_function(h, m, 40), 
    greedy_hunter_agent, 
    monster_agent, 
    arena_size=40, num_runs=100
)

# Bảng tổng hợp kết quả thử nghiệm
df_results = pd.DataFrame([
    {"Scenario": "Greedy vs Random (20x20)", "Hunter 1 Wins": exp1_res["1"], "Hunter 2 Wins": exp1_res["2"], "Monster Wins": exp1_res["M"], "Avg Steps": exp1_steps},
    {"Scenario": "Smart vs Greedy (30x30)", "Hunter 1 Wins": exp2_res["1"], "Hunter 2 Wins": exp2_res["2"], "Monster Wins": exp2_res["M"], "Avg Steps": exp2_steps},
    {"Scenario": "Smart vs Smart Monster (40x40)", "Hunter 1 Wins": exp3_res["1"], "Hunter 2 Wins": exp3_res["2"], "Monster Wins": exp3_res["M"], "Avg Steps": exp3_steps}
])

display(df_results)

# Biểu đồ trực quan hóa
sns.set_theme(style="whitegrid")
df_results.set_index("Scenario")[["Hunter 1 Wins", "Hunter 2 Wins", "Monster Wins"]].plot(
    kind="bar", stacked=True, figsize=(10, 5), colormap="viridis"
)
plt.title("GridHunt Multi-Agent Tournament Results (100 Runs)")
plt.ylabel("Number of Wins")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

[greedy] hunter=(2, 3) monster=(5, 1) dr=3 dc=-2 -> action='south'
[smart] hunter=(2, 3) monster=(5, 1) dist=5 dr=3 dc=-2 -> action='south'


TypeError: run_exp() got multiple values for argument 'arena_size'

## Your Conclusion [4 points] 

Discuss the following:

* What is your hunter's final strategy to choose actions. Why does it work well?
* Do you use teleportation. Why and in what situation? Why not?
* How does the arena size affects your hunter agent's performance?

> Your discussion goes here

The monster is also an agent. Describe how the monster's agent function could be changed so it gets better at avoiding the hunter.

> Your discussion goes here

# More Work (optional)

* Improve the monster agent so it tries to run away from the hunters. Don;t make the monster too fast or the hunters will never be able to catch it.
* How could the two hunter agents work together? You need to think about information sharing. This can be implemented by the two agent functions sharing a single Python class or the environment could be expanded by an action "shout" that lets the other agent perceive a message.